# Archived Databricks example

Historical source and outputs, not locally executable. Nothing was run during conversion. Databricks services, datasets and original image assets may be unavailable; image references are preserved, not downloaded. Outputs and errors are archived, not new execution.

## Transform Payments Data
1. Extract Date and Time from payment_timestamp and create new columns payment_date and payment_time
1. Map payment_status to contain descriptive values  
   (1-Success, 2-Pending, 3-Cancelled, 4-Failed)
1. Write transformed data to the Silver schema 

In [ ]:
df = spark.table('gizmobox.bronze.py_payments')
display(df)

Archived table:
payment_id	order_id	payment_timestamp	payment_status	payment_method
34	12	2024-12-06T22:30:12Z	1	PayPal
35	13	2024-12-17T05:25:34Z	2	PayPal
36	16	2024-12-09T10:45:56Z	4	Bank Transfer
37	17	2024-12-02T13:50:20Z	1	Bank Transfer
38	24	2024-12-22T08:30:55Z	1	PayPal
39	27	2024-12-03T11:45:30Z	2	Credit Card
40	28	2024-12-27T05:40:10Z	4	Credit Card
41	32	2024-12-29T09:20:40Z	1	PayPal
42	37	2024-12-23T12:10:05Z	2	Credit Card
43	38	2024-12-26T20:15:15Z	1	Credit Card
44	44	2024-12-21T14:25:45Z	1	Credit Card
45	49	2024-12-08T16:40:35Z	4	PayPal
46	54	2024-12-21T13:10:25Z	2	Credit Card
47	58	2024-12-10T11:50:20Z	1	PayPal
48	60	2024-12-14T18:30:15Z	1	Bank Transfer
49	61	2024-12-08T14:45:50Z	2	Bank Transfer
50	65	2024-12-10T12:35:05Z	4	Bank Transfer
51	67	2024-12-20T09:40:25Z	1	PayPal
52	69	2024-12-31T10:15:50Z	1	Bank Transfer
53	70	2024-12-05T13:50:35Z	2	PayPal
54	71	2024-12-28T11:05:40Z	1	Bank Transfer
55	74	2024-12-23T14:25:25Z	1	PayPal
56	80	2024-12-12T08:30:55Z	4	Bank Transfer
57

### 1. Extract Date and Time from payment_timestamp

In [ ]:
from pyspark.sql import functions as F

df_extracted_payments = (
                            df
                            .select (
                                'payment_id',
                                'order_id',
                                F.date_format('payment_timestamp','yyyy-MM-dd').cast('date').alias('payment_date'),
                                F.date_format('payment_timestamp','HH-mm-ss').alias('payment_time'),
                                'payment_status',
                                'payment_method'
                            )
)
display (df_extracted_payments)

Archived table:
payment_id	order_id	payment_date	payment_time	payment_status	payment_method
34	12	2024-12-06	22-30-12	1	PayPal
35	13	2024-12-17	05-25-34	2	PayPal
36	16	2024-12-09	10-45-56	4	Bank Transfer
37	17	2024-12-02	13-50-20	1	Bank Transfer
38	24	2024-12-22	08-30-55	1	PayPal
39	27	2024-12-03	11-45-30	2	Credit Card
40	28	2024-12-27	05-40-10	4	Credit Card
41	32	2024-12-29	09-20-40	1	PayPal
42	37	2024-12-23	12-10-05	2	Credit Card
43	38	2024-12-26	20-15-15	1	Credit Card
44	44	2024-12-21	14-25-45	1	Credit Card
45	49	2024-12-08	16-40-35	4	PayPal
46	54	2024-12-21	13-10-25	2	Credit Card
47	58	2024-12-10	11-50-20	1	PayPal
48	60	2024-12-14	18-30-15	1	Bank Transfer
49	61	2024-12-08	14-45-50	2	Bank Transfer
50	65	2024-12-10	12-35-05	4	Bank Transfer
51	67	2024-12-20	09-40-25	1	PayPal
52	69	2024-12-31	10-15-50	1	Bank Transfer
53	70	2024-12-05	13-50-35	2	PayPal
54	71	2024-12-28	11-05-40	1	Bank Transfer
55	74	2024-12-23	14-25-25	1	PayPal
56	80	2024-12-12	08-30-55	4	Bank Transfer
57	81	2024-12-28	

### 2. Map payment_status to contain descriptive values  
   (1-Success, 2-Pending, 3-Cancelled, 4-Failed)

In [ ]:
df_mapped_payments = (
                        df_extracted_payments
                        .select (
                            'payment_id',
                            'order_id',
                            'payment_date',
                            'payment_time',
                            F.when(df_extracted_payments.payment_status == 1, 'Success')
                             .when(df_extracted_payments.payment_status == 2, 'Pending')
                             .when(df_extracted_payments.payment_status == 3, 'Cancelled')
                             .when(df_extracted_payments.payment_status == 4, 'Failed')
                             .alias('payment_status'),
                             'payment_method'
                             )
)
display(df_mapped_payments)                        

Archived table:
payment_id	order_id	payment_date	payment_time	payment_status	payment_method
34	12	2024-12-06	22-30-12	Success	PayPal
35	13	2024-12-17	05-25-34	Pending	PayPal
36	16	2024-12-09	10-45-56	Failed	Bank Transfer
37	17	2024-12-02	13-50-20	Success	Bank Transfer
38	24	2024-12-22	08-30-55	Success	PayPal
39	27	2024-12-03	11-45-30	Pending	Credit Card
40	28	2024-12-27	05-40-10	Failed	Credit Card
41	32	2024-12-29	09-20-40	Success	PayPal
42	37	2024-12-23	12-10-05	Pending	Credit Card
43	38	2024-12-26	20-15-15	Success	Credit Card
44	44	2024-12-21	14-25-45	Success	Credit Card
45	49	2024-12-08	16-40-35	Failed	PayPal
46	54	2024-12-21	13-10-25	Pending	Credit Card
47	58	2024-12-10	11-50-20	Success	PayPal
48	60	2024-12-14	18-30-15	Success	Bank Transfer
49	61	2024-12-08	14-45-50	Pending	Bank Transfer
50	65	2024-12-10	12-35-05	Failed	Bank Transfer
51	67	2024-12-20	09-40-25	Success	PayPal
52	69	2024-12-31	10-15-50	Success	Bank Transfer
53	70	2024-12-05	13-50-35	Pending	PayPal
54	71	2024-12-28	11-

Historical archived error:
{
  "baseErrorDetails": {
    "addedWidgets": {},
    "arguments": {},
    "datasetInfos": [],
    "jupyterProps": {
      "ename": "SyntaxError",
      "evalue": "'(' was never closed (command-8182916970610204-3824660075, line 1)"
    },
    "metadata": {},
    "removedWidgets": [],
    "sqlProps": null,
    "stackFrames": [
      "\u001b  File \u001b<command-8182916970610204>, line 1\u001b\u001b\n\u001b    df_mapped_payments = (\u001b\n\u001b                         ^\u001b\n\u001bSyntaxError\u001b\u001b:\u001b '(' was never closed\n"
    ],
    "type": "baseError"
  }
}

### 3. Write transformed data to the Silver schema 

In [ ]:
df_mapped_payments.writeTo('gizmobox.silver.py_payments').createOrReplace()

Archived result list: empty.

Archived Databricks source — shown for reading, not Python execution.

```sql
%sql
SELECT * FROM gizmobox.silver.payments;
```

In [ ]:
# Archived output only; no computation.

Archived table:
payment_id	order_id	payment_date	payment_time	payment_status	payment_method
65	2	2025-01-19	05:05:13	Pending	PayPal
66	3	2025-01-08	23:45:00	Failed	Bank Transfer
67	4	2025-01-05	10:49:26	Success	Bank Transfer
68	8	2025-01-02	16:11:15	Pending	PayPal
69	14	2025-01-01	22:22:26	Failed	PayPal
70	18	2025-01-22	23:12:08	Success	Credit Card
71	22	2025-01-01	18:49:19	Success	PayPal
72	26	2025-01-05	22:01:25	Pending	Bank Transfer
73	29	2025-01-18	05:05:04	Failed	Bank Transfer
74	34	2025-01-12	11:30:15	Success	Credit Card
75	36	2025-01-16	23:15:43	Pending	PayPal
76	40	2025-01-08	13:22:51	Failed	Bank Transfer
77	45	2025-01-03	15:10:25	Success	PayPal
78	47	2025-01-11	20:20:20	Success	Bank Transfer
79	50	2025-01-17	12:30:00	Pending	Bank Transfer
80	53	2025-01-15	07:40:19	Failed	Credit Card
81	58	2025-01-04	18:49:33	Success	PayPal
82	60	2025-01-20	14:10:25	Pending	Bank Transfer
83	65	2025-01-06	20:30:10	Failed	PayPal
84	67	2025-01-14	16:00:00	Success	Bank Transfer
85	71	2025-01-07	19: